## Modern Embedding : BERT

BERT (Bidirectional Encoder Representations from Transformers) est un modèle de traitement automatique du langage (NLP) développé par Google en 2018. Ce modèle préentraîné révolutionne la compréhension du langage naturel en capturant le contexte des mots dans les deux directions d’une phrase, améliorant ainsi la précision des systèmes d’analyse linguistique. Il est devenu une référence pour de nombreuses tâches de NLP telles que la classification, la traduction, la recherche ou le question-réponse.

In [ ]:
import pandas as pd
import csv
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA n'est pas disponible. Verifiez les drivers NVIDIA, la presence du GPU et l'environnement du notebook."
    )

device = torch.device("cuda")
print(f"Device utilise: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Ouvre le CSV ../data/reddit_depression_dataset_cleaned_v2.csv
df = pd.read_csv("../data/reddit_depression_dataset_cleaned_v2.csv")

In [ ]:
df.head()

In [ ]:
# Prépare les données pour l'entraînement
df = df.dropna(subset=["label"]).copy()
texts = df["clean_text"].fillna("").astype(str).tolist()
labels = df["label"].tolist()

In [ ]:
# 1. Split (même que LogisticRegression)
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

In [ ]:
# modele BERT
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")
model.to(device)
model.eval()


def normalize_text(x):
    if x is None:
        return ""
    if isinstance(x, str):
        return x
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    return str(x)


def embed(texts, batch_size=32):
    embeddings = []

    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_raw = texts[i : i + batch_size]
            batch_texts = [normalize_text(t) for t in batch_raw]
            inputs = tokenizer(
                batch_texts,
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=512,
            )
            inputs = {k: v.to(device) for k, v in inputs.items()}
            outputs = model(**inputs)

            # moyenne des embeddings
            last_hidden = outputs.last_hidden_state
            emb = last_hidden.mean(dim=1).cpu().numpy()
            embeddings.append(emb)

    return np.vstack(embeddings)


# embeddings
X_train_emb = embed(X_train)
X_test_emb = embed(X_test)

# modele classique dessus
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_emb, y_train)

# evaluation
y_pred = clf.predict(X_test_emb)
print(classification_report(y_test, y_pred))